In [12]:
import re
import json
import os

from pathlib import Path
from collections import defaultdict
from datetime import datetime

CORPUS_DIR = Path("corpus/zoning")
OUTPUT_PATH = Path("chroma_db/section_graph.json")

print("Config loaded.")
print("Corpus dir exists:", CORPUS_DIR.exists())
print("Corpus path:", CORPUS_DIR.resolve())


Config loaded.
Corpus dir exists: True
Corpus path: D:\sk\planso_assignment\corpus\zoning


In [13]:
raw_files = {}

for file_path in sorted(CORPUS_DIR.glob("*.md")):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_files[file_path.name] = f.read()

print(f"Total files loaded: {len(raw_files)}")
print("\nFiles:\n")

for filename in raw_files:
    print("-", filename)

Total files loaded: 10

Files:

- zr_01_rules_of_construction.md
- zr_02_definitions_key.md
- zr_03_rear_yard_requirements.md
- zr_04_permitted_obstructions_rear_yard.md
- zr_05_floor_area_R6_R12_current.md
- zr_06_floor_area_R6_R10_SUPERSEDED_2019.md
- zr_07_front_yard_requirements.md
- zr_08_permitted_obstructions_all_yards.md
- zr_09_ceqr_e_designations.md
- zr_10_height_setback_R6_R12.md


In [14]:
def parse_document(filename: str, text: str):
    lines = text.splitlines()

    # =========================================================
    # HEADER PARSING
    # =========================================================

    document_title = ""
    last_amended = ""

    for line in lines[:10]:

        if line.startswith("# "):
            document_title = line.replace("# ", "").strip()

        if "**Last Amended:**" in line:
            last_amended = (
                line.replace("**Last Amended:**", "")
                .strip()
            )

    # =========================================================
    # HELPERS
    # =========================================================

    nodes = []

    current_section = None
    current_section_idx = None

    section_positions = []

    for idx, line in enumerate(lines):

        if line.startswith("## "):

            match = re.search(r"\b(\d{2}-\d{2,3})\b", line)

            if match:
                section_id = match.group(1)
            
            else:
                # -------------------------------------------------
                # FALLBACK FOR APPENDIX / TABLE / NON-STANDARD DOCS
                # -------------------------------------------------
            
                synthetic_idx = len(section_positions)
            
                section_id = (
                    Path(filename).stem
                    .replace("zr_", "")
                    .replace("-", "_")
                )
            
                section_id = f"{section_id}_section_{synthetic_idx}"
            
                print(
                    f"[INFO] Using synthetic section_id "
                    f"'{section_id}' for: {line}"
                )
            
            section_positions.append(
                {
                    "line_idx": idx,
                    "section_id": section_id,
                    "section_title": line.strip()
                }
            )

    # =========================================================
    # PROCESS EACH ## SECTION
    # =========================================================

    for sec_idx, section in enumerate(section_positions):

        section_id = section["section_id"]
        section_title = section["section_title"]

        start_idx = section["line_idx"]

        if sec_idx + 1 < len(section_positions):
            end_idx = section_positions[sec_idx + 1]["line_idx"]
        else:
            end_idx = len(lines)

        section_lines = lines[start_idx:end_idx]

        # -----------------------------------------------------
        # Find ### subsections
        # -----------------------------------------------------

        subsection_positions = []

        for local_idx, line in enumerate(section_lines):

            if line.startswith("### "):

                subsection_match = re.search(
                    r"\b(\d{2}-\d{3}[a-z]?)\b",
                    line
                )

                if subsection_match:
                    subsection_id = subsection_match.group(1)

                else:
                    subsection_id = f"{section_id}_{len(subsection_positions)}"

                subsection_positions.append(
                    {
                        "subsection_id": subsection_id,
                        "subsection_title": line.strip(),
                        "line_idx": start_idx + local_idx
                    }
                )

        # -----------------------------------------------------
        # CASE 1: NO ### subsections
        # -----------------------------------------------------

        if len(subsection_positions) == 0:

            subsection_id = f"{section_id}_body"

            body_text = "\n".join(section_lines[1:]).strip()

            refs = re.findall(
                r"\bSection\s+(\d{2}-\d{2,3}[a-z]?)\b",
                body_text
            )

            refs += re.findall(
                r"\b(\d{2}-\d{3}[a-z]?)\b",
                body_text
            )

            refs = list(set(refs))

            refs = [
                r for r in refs
                if r not in [subsection_id, section_id]
            ]

            has_table = ("|" in body_text and "---" in body_text)

            node = {
                "node_id": subsection_id,
                "section_id": section_id,
                "section_title": section_title,
                "subsection_title": section_title,
                "source_file": filename,
                "document_title": document_title,
                "last_amended": last_amended,
                "start_line": start_idx,
                "end_line": end_idx - 1,
                "has_table": has_table,
                "has_cross_ref": len(refs) > 0,
                "references": refs,
                "referenced_by": [],
                "body_text": body_text
            }

            nodes.append(node)

        # -----------------------------------------------------
        # CASE 2: HAS ### subsections
        # -----------------------------------------------------

        else:

            for sub_idx, subsection in enumerate(subsection_positions):

                subsection_id = subsection["subsection_id"]
                subsection_title = subsection["subsection_title"]

                sub_start = subsection["line_idx"]

                if sub_idx + 1 < len(subsection_positions):
                    sub_end = subsection_positions[sub_idx + 1]["line_idx"]
                else:
                    sub_end = end_idx

                body_lines = lines[sub_start + 1:sub_end]

                body_text = "\n".join(body_lines).strip()

                refs = re.findall(
                    r"\bSection\s+(\d{2}-\d{2,3}[a-z]?)\b",
                    body_text
                )

                refs += re.findall(
                    r"\b(\d{2}-\d{3}[a-z]?)\b",
                    body_text
                )

                refs = list(set(refs))

                refs = [
                    r for r in refs
                    if r not in [subsection_id, section_id]
                ]

                has_table = ("|" in body_text and "---" in body_text)

                node = {
                    "node_id": subsection_id,
                    "section_id": section_id,
                    "section_title": section_title,
                    "subsection_title": subsection_title,
                    "source_file": filename,
                    "document_title": document_title,
                    "last_amended": last_amended,
                    "start_line": sub_start,
                    "end_line": sub_end - 1,
                    "has_table": has_table,
                    "has_cross_ref": len(refs) > 0,
                    "references": refs,
                    "referenced_by": [],
                    "body_text": body_text
                }

                nodes.append(node)

    return nodes


print("Parser defined.")

Parser defined.


In [15]:
all_nodes = []

for filename, text in raw_files.items():

    nodes = parse_document(filename, text)

    all_nodes.extend(nodes)

    print(f"{filename}: {len(nodes)} chunks")

print("\n===================================================")
print(f"Total chunks: {len(all_nodes)}")

print(
    f"Chunks with cross-refs: "
    f"{sum(1 for n in all_nodes if n['has_cross_ref'])}"
)

print(
    f"Chunks with tables: "
    f"{sum(1 for n in all_nodes if n['has_table'])}"
)

# =====================================================
# SAMPLE NODE
# =====================================================

sample_node = {
    k: v
    for k, v in all_nodes[0].items()
    if k != "body_text"
}

print("\nSample node:\n")

print(json.dumps(sample_node, indent=2))

zr_01_rules_of_construction.md: 2 chunks
zr_02_definitions_key.md: 4 chunks
zr_03_rear_yard_requirements.md: 3 chunks
zr_04_permitted_obstructions_rear_yard.md: 1 chunks
zr_05_floor_area_R6_R12_current.md: 1 chunks
zr_06_floor_area_R6_R10_SUPERSEDED_2019.md: 1 chunks
zr_07_front_yard_requirements.md: 2 chunks
zr_08_permitted_obstructions_all_yards.md: 1 chunks
[INFO] Using synthetic section_id '09_ceqr_e_designations_section_0' for: ## City Environmental Quality Review (CEQR): (E) Designations — Excerpt
[INFO] Using synthetic section_id '09_ceqr_e_designations_section_1' for: ## What is an (E) Designation?
[INFO] Using synthetic section_id '09_ceqr_e_designations_section_2' for: ## How to Use This Table
[INFO] Using synthetic section_id '09_ceqr_e_designations_section_3' for: ## Sample Records (Illustrative — not a complete listing)
zr_09_ceqr_e_designations.md: 4 chunks
zr_10_height_setback_R6_R12.md: 3 chunks

Total chunks: 22
Chunks with cross-refs: 10
Chunks with tables: 6

Sample 

In [16]:
# =====================================================
# BUILD NODE LOOKUP
# =====================================================

node_lookup = {
    node["node_id"]: node
    for node in all_nodes
}

# =====================================================
# BUILD GRAPH EDGES
# =====================================================

in_corpus_refs = set()
missing_refs = set()

for node in all_nodes:

    for ref_id in node["references"]:

        if ref_id in node_lookup:

            # ---------------------------------------------
            # Avoid duplicate referenced_by entries
            # ---------------------------------------------

            if node["node_id"] not in node_lookup[ref_id]["referenced_by"]:

                node_lookup[ref_id]["referenced_by"].append(
                    node["node_id"]
                )

            in_corpus_refs.add(
                (node["node_id"], ref_id)
            )

        else:

            missing_refs.add(ref_id)

# =====================================================
# GRAPH STATS
# =====================================================

total_edges = sum(
    len(node["references"])
    for node in all_nodes
)

print("Graph built.\n")

print("Total edges:", total_edges)

print(
    "References inside corpus:",
    len(in_corpus_refs)
)

print(
    "References outside corpus:",
    len(missing_refs)
)

# =====================================================
# TOP REFERENCED NODES
# =====================================================

sorted_nodes = sorted(
    all_nodes,
    key=lambda x: len(x["referenced_by"]),
    reverse=True
)

print("\nTop 5 most referenced nodes:\n")

for node in sorted_nodes[:5]:

    print(
        f"{node['node_id']} "
        f"← referenced by "
        f"{len(node['referenced_by'])} nodes"
    )

# =====================================================
# SHOW CORPUS GAPS
# =====================================================

print("\nCorpus gaps (missing refs):\n")

for ref in sorted(list(missing_refs))[:15]:
    print("-", ref)

Graph built.

Total edges: 18
References inside corpus: 4
References outside corpus: 12

Top 5 most referenced nodes:

23-342 ← referenced by 1 nodes
23-344 ← referenced by 1 nodes
23-431 ← referenced by 1 nodes
23-433 ← referenced by 1 nodes
12-01_body ← referenced by 0 nodes

Corpus gaps (missing refs):

- 12-01
- 12-10
- 23-154
- 23-311
- 23-312
- 23-341
- 23-613
- 23-711
- 23-73
- 23-90
- 24-01
- 26-50


In [17]:
os.makedirs("chroma_db", exist_ok=True)

graph_output = {
    "metadata": {
        "total_nodes": len(all_nodes),

        "total_edges": sum(
            len(n["references"])
            for n in all_nodes
        ),

        "corpus_gaps": list(set([
            ref
            for node in all_nodes
            for ref in node["references"]
            if ref not in node_lookup
        ])),

        "generated_at": datetime.now().isoformat()
    },

    "nodes": {

        node["node_id"]: {
            k: v
            for k, v in node.items()
            if k != "body_text"
        }

        for node in all_nodes
    }
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(graph_output, f, indent=2)

print("Written to chroma_db/section_graph.json")

print("Nodes:", len(graph_output["nodes"]))

print(
    "Edges:",
    graph_output["metadata"]["total_edges"]
)

print(
    "Corpus gaps:",
    len(set(graph_output["metadata"]["corpus_gaps"]))
)

Written to chroma_db/section_graph.json
Nodes: 21
Edges: 18
Corpus gaps: 12


In [18]:
chunks = []

for node in all_nodes:

    chunk = {
        "chunk_id": f"{node['source_file']}::{node['node_id']}",

        "node_id": node["node_id"],

        "section_id": node["section_id"],

        "section_title": node["section_title"],

        "subsection_title": node["subsection_title"],

        "source_file": node["source_file"],

        "last_amended": node["last_amended"],

        "start_line": node["start_line"],

        "end_line": node["end_line"],

        "has_table": node["has_table"],

        "has_cross_ref": node["has_cross_ref"],

        "references": node["references"],

        "text": (
            f"{node['section_title']}\n"
            f"{node['subsection_title']}\n\n"
            f"{node['body_text']}"
        )
    }

    chunks.append(chunk)

chunk_output = {
    "chunks": chunks
}

with open(
    "chroma_db/subsection_chunks.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(chunk_output, f, indent=2)

print("Chunks written:", len(chunks))
print("File: chroma_db/subsection_chunks.json")

Chunks written: 22
File: chroma_db/subsection_chunks.json


Stage 2 indexing

In [20]:
!pip install rank-bm25

In [21]:
import json
import pickle
import re

from pathlib import Path

import chromadb

from chromadb.utils.embedding_functions import (
    SentenceTransformerEmbeddingFunction
)

from rank_bm25 import BM25Okapi

# =====================================================
# CONFIG
# =====================================================

CHUNKS_PATH = Path("chroma_db/subsection_chunks.json")

CHROMA_PATH = "chroma_db"

COLLECTION = "zoning_docs"

EMBED_MODEL = "all-MiniLM-L6-v2"

BM25_PATH = Path("chroma_db/bm25_index.pkl")

# =====================================================
# LOAD CHUNKS
# =====================================================

chunks = json.loads(
    CHUNKS_PATH.read_text(encoding="utf-8")
)["chunks"]

print(f"Loaded {len(chunks)} chunks")

print("\nSample chunk_id:")
print(chunks[0]["chunk_id"])

Loaded 22 chunks

Sample chunk_id:
zr_01_rules_of_construction.md::12-01_body


In [22]:
# =====================================================
# EMBEDDING FUNCTION
# =====================================================

ef = SentenceTransformerEmbeddingFunction(
    model_name=EMBED_MODEL
)

# =====================================================
# CHROMA CLIENT
# =====================================================

client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

# =====================================================
# DELETE OLD COLLECTION
# =====================================================

try:

    client.delete_collection(COLLECTION)

    print("Deleted existing collection")

except Exception:

    print("No existing collection found")

# =====================================================
# CREATE COLLECTION
# =====================================================

collection = client.create_collection(
    COLLECTION,
    embedding_function=ef
)

print(f"Created collection: {COLLECTION}")

# =====================================================
# INSERT IN BATCHES
# =====================================================

BATCH = 50

for i in range(0, len(chunks), BATCH):

    batch = chunks[i:i+BATCH]

    collection.add(

        ids=[
            c["chunk_id"]
            for c in batch
        ],

        documents=[

            (
                f"{c['node_id']}\n"
                f"{c['section_title']}\n"
                f"{c['subsection_title']}\n\n"
                f"{c['text']}"
            )

            for c in batch
        ],

        metadatas=[

            {

                "node_id": c["node_id"],

                "section_id": c["section_id"],

                "section_title": c["section_title"],

                "subsection_title": c["subsection_title"],

                "source_file": c["source_file"],

                "last_amended": c["last_amended"],

                "start_line": c["start_line"],

                "end_line": c["end_line"],

                "has_table": c["has_table"],

                "has_cross_ref": c["has_cross_ref"],

                "references": json.dumps(
                    c["references"]
                ),

                "chunk_type": (
                    "appendix"
                    if "section_" in c["node_id"]
                    else "zoning"
                )

            }

            for c in batch
        ]
    )

    print(
        f"Added batch {i//BATCH + 1} "
        f"({i} → {min(i+BATCH, len(chunks))})"
    )

print("\nDense index complete")

print(
    "Total chunks in collection:",
    collection.count()
)

No existing collection found
Created collection: zoning_docs
Added batch 1 (0 → 22)

Dense index complete
Total chunks in collection: 22


In [23]:
test_results = collection.query(

    query_texts=[
        "rear yard requirements depth feet"
    ],

    n_results=3,

    include=[
        "metadatas",
        "distances"
    ]
)

print(
    "Test query:\n"
    "'rear yard requirements depth feet'\n"
)

for meta, dist in zip(

    test_results["metadatas"][0],

    test_results["distances"][0]

):

    similarity = round(1 - dist, 3)

    print(
        f"[{similarity}] "
        f"{meta['source_file']} "
        f":: "
        f"{meta['subsection_title'][:80]}"
    )

Test query:
'rear yard requirements depth feet'

[0.73] zr_03_rear_yard_requirements.md :: ### 23-343: Rear Yard Equivalent Requirements
[0.715] zr_07_front_yard_requirements.md :: ### 23-321: Basic Front Yard Requirements in R1 Through R5 Districts
[0.683] zr_03_rear_yard_requirements.md :: ### 23-342: Rear Yard Requirements


In [24]:
# =====================================================
# TOKENIZER
# =====================================================

def tokenize(text: str) -> list:

    text = text.lower()

    return re.findall(
        r'[a-z0-9\-\(\)]+',
        text
    )

# =====================================================
# BUILD CORPUS
# =====================================================

corpus_ids = [
    c["chunk_id"]
    for c in chunks
]

corpus_texts = [

    tokenize(

        (
            f"{c['node_id']} "
            f"{c['section_title']} "
            f"{c['subsection_title']} "
            f"{c['text']}"
        )

    )

    for c in chunks
]

# =====================================================
# BUILD BM25
# =====================================================

bm25 = BM25Okapi(corpus_texts)

print(
    f"BM25 index built over "
    f"{len(corpus_ids)} documents"
)

# =====================================================
# SAVE
# =====================================================

payload = {

    "bm25": bm25,

    "corpus_ids": corpus_ids

}

with open(BM25_PATH, "wb") as f:

    pickle.dump(payload, f)

print(f"BM25 index saved to:\n{BM25_PATH}")

BM25 index built over 22 documents
BM25 index saved to:
chroma_db\bm25_index.pkl


In [25]:
test_query = tokenize(
    "air conditioning unit rear yard obstruction"
)

scores = bm25.get_scores(test_query)

top5 = sorted(

    range(len(scores)),

    key=lambda i: scores[i],

    reverse=True

)[:5]

print(
    "Test query:\n"
    "'air conditioning unit rear yard obstruction'\n"
)

for idx in top5:

    chunk = chunks[idx]

    print(

        f"[{round(scores[idx], 3)}] "

        f"{chunk['source_file']} "

        f":: "

        f"{chunk['subsection_title'][:80]}"
    )

Test query:
'air conditioning unit rear yard obstruction'

[8.104] zr_04_permitted_obstructions_rear_yard.md :: ## Section 23-341: Permitted Obstructions in Required Rear Yards or Rear Yard Eq
[5.027] zr_08_permitted_obstructions_all_yards.md :: ## Section 23-311: Permitted Obstructions in All Yards, Courts and Open Areas
[3.795] zr_03_rear_yard_requirements.md :: ### 23-343: Rear Yard Equivalent Requirements
[3.763] zr_03_rear_yard_requirements.md :: ### 23-342: Rear Yard Requirements
[3.709] zr_03_rear_yard_requirements.md :: ### 23-344: Additional Rear Yard Modifications


In [26]:
query = "floor area ratio maximum R7A district"

print("=" * 120)
print("QUERY:")
print(query)
print("=" * 120)

# =====================================================
# DENSE RESULTS
# =====================================================

dense_results = collection.query(

    query_texts=[query],

    n_results=5,

    include=["metadatas"]

)

dense_chunks = dense_results["metadatas"][0]

# =====================================================
# BM25 RESULTS
# =====================================================

scores = bm25.get_scores(
    tokenize(query)
)

top5 = sorted(

    range(len(scores)),

    key=lambda i: scores[i],

    reverse=True

)[:5]

bm25_chunks = [
    chunks[i]
    for i in top5
]

# =====================================================
# COMPARISON TABLE
# =====================================================

print(
    f"\n{'Rank':<6}"
    f"{'Dense Retrieval':<55}"
    f"{'BM25 Retrieval'}"
)

print("-" * 120)

for rank in range(5):

    dense_text = (
        f"{dense_chunks[rank]['source_file']} :: "
        f"{dense_chunks[rank]['subsection_title'][:40]}"
    )

    bm25_text = (
        f"{bm25_chunks[rank]['source_file']} :: "
        f"{bm25_chunks[rank]['subsection_title'][:40]}"
    )

    print(
        f"{rank+1:<6}"
        f"{dense_text:<55}"
        f"{bm25_text}"
    )

QUERY:
floor area ratio maximum R7A district

Rank  Dense Retrieval                                        BM25 Retrieval
------------------------------------------------------------------------------------------------------------------------
1     zr_05_floor_area_R6_R12_current.md :: ## Section 23-22: Floor Area Regulationszr_05_floor_area_R6_R12_current.md :: ## Section 23-22: Floor Area Regulations
2     zr_06_floor_area_R6_R10_SUPERSEDED_2019.md :: ## Section 23-22: Floor Area Regulationszr_06_floor_area_R6_R10_SUPERSEDED_2019.md :: ## Section 23-22: Floor Area Regulations
3     zr_10_height_setback_R6_R12.md :: ### 23-432: Height and Setback Requiremezr_02_definitions_key.md :: ### floor area
4     zr_02_definitions_key.md :: ### floor area             zr_10_height_setback_R6_R12.md :: ### 23-433: Standard Setback Regulations
5     zr_07_front_yard_requirements.md :: ### 23-321: Basic Front Yard Requirementzr_10_height_setback_R6_R12.md :: ### 23-432: Height and Setback Requireme